# Paraguay · GOES-East (G19) · infrarrojo limpio C13 (10.3 µm)

Animación con zoom en **Paraguay** usando archivos reales
`OR_ABI-L2-CMIPF-M6C13_G19_*.nc` (NOAA/NESDIS, bucket público `noaa-goes19`).

- **Satélite y banda**: GOES-East (G19, en 75.2°O), banda **C13 (10.33 µm)**,
  temperatura de brillo en °C. Es la banda "Clean Infrared": ve nubes de día y de noche.
- **Encuadre**: recorte a la zona Paraguay (oeste -63, este -53, sur -30, norte -18),
  a resolución completa (sin submuestreo).
- **Escala**: percentiles automáticos sobre el primer cuadro, igual que `poster_goes.py`;
  paleta `termico` (frío = blanco, como en las imágenes IR de siempre).
- **Cadencia**: 10 min (modo 6); el nombre del archivo la trae escrita.
- **Vestuario**: el mismo de `poster_goes.py` (`draw_goes_poster` + `caption_lines`).


In [ ]:
# ==== configuración ====
from datetime import datetime
from pathlib import Path

# Raíz del proyecto, se corra el notebook desde notebooks/ o desde la raíz.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

T0       = datetime(2026, 10, 3, 22, 20)   # primer cuadro (UTC)
N_FRAMES = 12                              # cantidad de cuadros
PASO_MIN = 10                              # cadencia real de C13 en modo 6

SAT, BANDA = "G19", "C13"                  # GOES-East, IR limpio 10.3 µm
PALETA     = "termico"                     # gris invertido: frío = blanco
CREDITO    = "Graphic by @tu_usuario"      # tu handle: arriba a la derecha + marca de agua
ZONA       = "py"                          # Paraguay y alrededores (ver ZONES en poster_goes.py)

DESCARGAR  = True                          # False = solo archivos locales; falla si falta un cuadro

# Escala de la barra: None = percentiles automáticos, igual que poster_goes.py.
VMIN, VMAX = None, None
ESCALA_PERCENTILES = (0.5, 99.5)
STRIDE = 1                                 # 1 = resolución completa (el recorte es chico)

LADO_PULG, DPI = 6.0, 120                  # 720 px por cuadro de la animación
CUADRO_MS, FUNDIDO_MS, FUNDIDO_PASOS = 300, 100, 3


In [ ]:
# ==== datos y utilidades ====
import io, re, sys, urllib.request
from collections import defaultdict
from datetime import timedelta
from pathlib import Path

import numpy as np, xarray as xr, cartopy.crs as ccrs
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as MostrarImagen, display
from rich.progress import track

sys.path.insert(0, str(RAIZ / "lib"))
from poster_goes import (BG, PALETTES, ZONES, box_indices, caption_lines,
                         draw_goes_poster, to_display)

DATOS = Path(r"D:\Users\guill\dataset era5\goes_c13_g19")
DATOS.mkdir(exist_ok=True)
CAJA = ZONES[ZONA]

POR_HORA = defaultdict(list)


def claves_de_la_hora(t):
    """Claves S3 de la hora de `t` (una sola consulta por hora)."""
    hora = t.replace(minute=0, second=0, microsecond=0)
    if hora not in POR_HORA:
        pref = (f"ABI-L2-CMIPF/{hora.year}/{int(hora.strftime('%j')):03d}/"
                f"{hora.hour:02d}/")
        url = f"https://noaa-goes{SAT[1:]}.s3.amazonaws.com/?list-type=2&prefix={pref}"
        xml = urllib.request.urlopen(url, timeout=120).read().decode()
        POR_HORA[hora] = re.findall(r"<Key>([^<]+)</Key>", xml)
    return POR_HORA[hora]


PATRON = re.compile(r"OR_ABI-L2-CMIPF-M6(?P<banda>C\d+)_(?P<sat>G\d+)_s(?P<s>\d{14})_")


def clave_de(t):
    """Clave del archivo cuyo escaneo arranca en el slot de 10 min de `t`."""
    for clave in claves_de_la_hora(t):
        m = PATRON.search(clave)
        if not m or m["banda"] != BANDA or m["sat"] != SAT:
            continue
        inicio = datetime.strptime(m["s"][:13], "%Y%j%H%M%S")
        if (inicio.hour, inicio.minute) == (t.hour, t.minute):
            return clave
    raise LookupError(f"Sin archivo {SAT} {BANDA} para {t:%Y-%m-%d %H:%M} UTC")


def ruta_local(clave):
    """Si el .nc ya está en la carpeta del proyecto, se reutiliza sin copiarlo."""
    nombre = clave.rsplit("/", 1)[-1]
    for candidato in (Path(nombre), DATOS / nombre):
        if candidato.exists():
            return candidato
    return DATOS / nombre


def descargar(clave, permitir_red=True):
    destino = ruta_local(clave)
    if destino.exists() and destino.stat().st_size > 1_000_000:
        return destino
    if not permitir_red:
        raise SystemExit(f"Falta {destino.name} y DESCARGAR=False: "
                         f"poné DESCARGAR=True para bajarlo de S3.")
    url = f"https://noaa-goes{SAT[1:]}.s3.amazonaws.com/{clave}"
    with urllib.request.urlopen(url, timeout=300) as respuesta:
        with open(destino, "wb") as salida:
            while True:
                trozo = respuesta.read(1 << 20)
                if not trozo:
                    break
                salida.write(trozo)
    return destino


def cargar(nc):
    """Recorte Paraguay a resolución completa: campo en °C, grilla, proyección y rótulos."""
    with xr.open_dataset(nc) as ds:
        cmi = ds["CMI"]
        gp = ds["goes_imager_projection"].attrs
        h = float(gp["perspective_point_height"])
        x = ds["x"].values[::STRIDE] * h
        y = ds["y"].values[::STRIDE] * h
        proj = ccrs.Geostationary(
            central_longitude=float(gp["longitude_of_projection_origin"]),
            satellite_height=h, sweep_axis=str(gp.get("sweep_angle_axis", "x")))
        i0, i1, j0, j1 = box_indices(proj, x, y, CAJA)
        crudo = np.asarray(cmi.values[j0:j1, i0:i1][::1, ::1], dtype=np.float64)
        campo, unidad, tipo = to_display(crudo, cmi)
        rotulos = caption_lines(ds, cmi, tipo, f"ZONA {ZONA.upper()}")
    return campo, x[i0:i1], y[j0:j1], proj, unidad, rotulos


SLOTS = [T0 + i * timedelta(minutes=PASO_MIN) for i in range(N_FRAMES)]
print(f"{len(SLOTS)} cuadros: {SLOTS[0]:%Y-%m-%d %H:%M} -> {SLOTS[-1]:%Y-%m-%d %H:%M} UTC")
print(f"zona {ZONA}: {CAJA}")


In [ ]:
# ==== descarga (solo lo que falta; nada si DESCARGAR=False y todo está local) ====
rutas = []
for t in track(SLOTS, description="Descargando cuadros"):
    ruta = descargar(clave_de(t), DESCARGAR)
    rutas.append(ruta)
    print(f"{t:%Y-%m-%d %H:%M} UTC  {ruta.name}  {ruta.stat().st_size / 1e6:.1f} MB")


In [ ]:
# ==== control: rótulos y recorte del primer cuadro ====
campo, x, y, proj, unidad, rotulos = cargar(rutas[0])
titulo, subtitulo, fuente, datos = rotulos
if VMIN is None or VMAX is None:
    lo, hi = np.nanpercentile(campo, ESCALA_PERCENTILES)
    VMIN = VMIN if VMIN is not None else float(lo)
    VMAX = VMAX if VMAX is not None else float(hi)
    print(f"escala automática {ESCALA_PERCENTILES}: {VMIN:.2f} a {VMAX:.2f} {unidad}")
print(f"recorte: {campo.shape} celdas")
print(titulo)
print(subtitulo)
print(fuente)
print(datos)
assert unidad == "°C"
assert titulo == "OCTOBER 2026"
assert subtitulo == "G19 GOES-East\nTEMPERATURA DE BRILLO · BANDA 13 (10.3 µm)"
assert datos == "DATA: G19 | 2026-10-03T22:25 UTC | MODO Mode 6"
assert fuente == "SOURCE: NOAA/NESDIS G19 ABI L2 CMIP | ZONA PY"
print("\nOK: rótulos y recorte verificados.")


In [ ]:
# ==== cuadro fijo (PNG) del primer instante ====
fig, ax, _ = draw_goes_poster(
    campo, x=x, y=y, proj=proj, title=titulo, subtitle=subtitulo, unit_label=unidad,
    src_line=fuente, data_line=datos, credit=CREDITO,
    cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
    xlim=(float(x.min()), float(x.max())), ylim=(float(y.min()), float(y.max())))
png = Path(f"poster_paraguay_{SAT}_{T0:%Y-%m-%dT%H%M}.png")
fig.savefig(png, facecolor=BG)
plt.close(fig)
print("PNG:", png)
display(MostrarImagen(filename=str(png)))


In [ ]:
# ==== render de todos los cuadros ====
cuadros = []
for i, ruta in enumerate(track(rutas, description="Renderizando"), 1):
    campo_i, x_i, y_i, proj_i, unidad_i, rot_i = cargar(ruta)
    fig, ax, _ = draw_goes_poster(
        campo_i, x=x_i, y=y_i, proj=proj_i, title=rot_i[0], subtitle=rot_i[1],
        unit_label=unidad_i, src_line=rot_i[2], data_line=rot_i[3],
        credit=CREDITO, cmap=PALETTES[PALETA], vmin=VMIN, vmax=VMAX,
        xlim=(float(x_i.min()), float(x_i.max())),
        ylim=(float(y_i.min()), float(y_i.max())),
        size=LADO_PULG, dpi=DPI)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", facecolor=BG)
    plt.close(fig)
    cuadros.append(Image.open(buffer).convert("RGB"))
    print(f"  cuadro {i}/{len(rutas)}  {rot_i[3]}")

print(f"{len(cuadros)} cuadros de {cuadros[0].size[0]}x{cuadros[0].size[1]} px")


In [ ]:
# ==== GIF con fundido entre cuadros ====
def con_fundido(cuadros, pasos, ms_cuadro, ms_fundido):
    """Pares (imagen, duración_ms): cuadros reales + intermedios interpolados."""
    salida = []
    for i, img in enumerate(cuadros):
        salida.append((img, ms_cuadro))
        if pasos and i + 1 < len(cuadros):
            a0 = np.asarray(img, dtype=np.float32)
            a1 = np.asarray(cuadros[i + 1], dtype=np.float32)
            for k in range(1, pasos + 1):
                alfa = k / (pasos + 1)
                mezcla = ((1 - alfa) * a0 + alfa * a1).astype(np.uint8)
                salida.append((Image.fromarray(mezcla), ms_fundido))
    return salida


fotogramas = con_fundido(cuadros, FUNDIDO_PASOS, CUADRO_MS, FUNDIDO_MS)
gif = Path(f"animacion_paraguay_{SAT}_{T0:%Y-%m-%dT%H%M}.gif")
fotogramas[0][0].save(
    gif, save_all=True,
    append_images=[im for im, _ in fotogramas[1:]],
    duration=[ms for _, ms in fotogramas], loop=0, optimize=True)
print(f"GIF: {gif}  ({len(fotogramas)} fotogramas, {gif.stat().st_size / 1e6:.1f} MB)")
display(MostrarImagen(filename=str(gif)))


## Ideas y variantes

- **Otra ventana**: cambiá `T0` y `N_FRAMES`; la clave S3 se arma sola a partir del
  slot de 10 minutos (no hace falta escribir el nombre del archivo).
- **Sin descargar**: `DESCARGAR = False` usa solo los .nc que ya estén en la carpeta
  (o en `goes_c13_g19/`); si falta un cuadro, avisa cuál en vez de bajarlo.
- **Otra zona**: `ZONA = "sa"` (Sudamérica), `"br"`, `"caribe"`, `"conus"`, o una caja
  propia pasando `CAJA = (oeste, este, sur, norte)` después de la celda de utilidades.
- **Escala fija**: poné `VMIN`/`VMAX` en °C para no depender del primer cuadro.
- **Sin fundido**: `FUNDIDO_PASOS = 0` deja la animación a saltos limpios de 10 min.
- **MP4**: en este entorno no hay `ffmpeg` en el PATH, por eso la salida es GIF.
- **Crédito**: `CREDITO` alimenta el texto de arriba a la derecha y la marca de agua
  central (esa se dibuja con las letras separadas).
